# DMA Engine Debugging & Error Handling

This notebook validates the DMA subsystem of the FIREQ Hardware Backend:
- DMA state machine inspection
- AXI Switch routing verification
- Error recovery mechanisms
- Timeout handling
- Zombie buffer cleanup (transactional safety)

## 1. Setup & Imports

In [1]:
import sys
import os
import numpy as np
import time
from pynq import PL, allocate

# Path setup - adjust based on your installation
sys.path.insert(0, os.path.join(os.getcwd(), 'fireq_utils'))

from backend import FireqHardwareBackend
from backend import (
    DMATimeoutError, 
    DMAError,
    TimingError, 
    ConfigurationError,
    DriverError
)

from FIREQ_LL_API.overlay_driver import FIREQ_SoC

BITSTREAM_PATH = "/home/xilinx/jupyter_notebooks/api_test_giorgio/fireq_ol/NEW_FIREQ.bit"
if not os.path.exists(BITSTREAM_PATH):
    raise FileNotFoundError(f"Bitstream not found at {BITSTREAM_PATH}!")

print("All imports successful!")

All imports successful!


In [2]:
# Clean slate - reset PL before loading
PL.reset()
print("PL reset complete.")

PL reset complete.


## 2. Load Overlay & Initialize Backend

In [3]:
print("--- LOADING OVERLAY ---")
overlay = FIREQ_SoC(BITSTREAM_PATH)
print("--- OVERLAY LOADED ---")

--- LOADING OVERLAY ---


--- OVERLAY LOADED ---


In [4]:
print("--- INIT BACKEND (debug=True) ---")
backend = FireqHardwareBackend(overlay, debug=True)
print(f"Backend ready. {len(backend.hw.gens)} Gen(s), {len(backend.hw.acqs)} Acq(s)")

[Backend] INFO: Initializing Backend...
[Backend] DEBUG: Step 1: Discovering hardware via HardwareInventory...
[Backend] DEBUG: HardwareInventory: Scanning overlay for IPs...
[Backend] DEBUG:   Phase 1: Discovering custom IPs...
[Backend] DEBUG:     [OK] Found 1 Signal Generator(s)
[Backend] DEBUG:     [OK] Found 2 Acquisition IP(s)
[Backend] DEBUG:     [OK] Found Trigger Generator
[Backend] DEBUG:   Phase 2: Discovering infrastructure IPs...
[Backend] DEBUG:     [OK] Found AXI DMA Controller
[Backend] DEBUG:     [OK] Found RF-DC Controller
[Backend] DEBUG:     [OK] Found AXI Switch
[Backend] DEBUG: Validating RF clocks and discovering sample rates...
[Backend] DEBUG:   Scanning DAC tiles...


--- INIT BACKEND (debug=True) ---


[Backend] DEBUG:     [OK] DAC Tile 0: 9.585 GSPS (Lock: 2)
[Backend] DEBUG:     [SKIP] DAC Tile 1: Not active or PLL read failed (RuntimeError)
[Backend] DEBUG:     [OK] DAC Tile 2: 9.585 GSPS (Lock: 2)
[Backend] DEBUG:     [SKIP] DAC Tile 3: Not active or PLL read failed (RuntimeError)
[Backend] DEBUG:   [OK] DAC sample rate: 9.585 GSPS
[Backend] DEBUG:   Scanning ADC tiles...
[Backend] DEBUG:     [OK] ADC Tile 0: 4.792 GSPS (Lock: 2)
[Backend] DEBUG:     [SKIP] ADC Tile 1: Not active or PLL read failed (RuntimeError)
[Backend] DEBUG:     [SKIP] ADC Tile 2: Not active or PLL read failed (RuntimeError)
[Backend] DEBUG:     [SKIP] ADC Tile 3: Not active or PLL read failed (RuntimeError)
[Backend] DEBUG:   [OK] ADC sample rate: 4.792 GSPS
[Backend] INFO: Clocks Validated. DAC: 9.585 GSPS, ADC: 4.792 GSPS
[Backend] DEBUG: Hardware specs populated: DAC 9.585G, ADC 4.792G
[Backend] DEBUG: Step 2: Creating GeneratorAdapters...
[Backend] DEBUG: Step 3: Creating AcquisitionAdapters...
[Backend

Backend ready. 1 Gen(s), 2 Acq(s)


## 3. DMA State Machine Inspection

The PYNQ DMA has specific state requirements:
- `running=True` AND `idle=True` before calling `.transfer()`
- PYNQ caches state that can become stale after hardware operations

**WARNING**: PYNQ's `.idle` and `.running` properties can BLOCK if DMA is stuck waiting for data.
We use MMIO-only reads when DMA might be in an uncertain state.

In [ ]:
def read_dma_status_mmio(dma_engine, safe_mode=False):
    """
    Read and decode DMA status register.
    
    Args:
        dma_engine: The AcquisitionEngine instance
        safe_mode: If True, only use MMIO (avoids PYNQ calls that might block)
    """
    dma = dma_engine.dma
    
    # Read S2MM (Stream-to-Memory-Mapped) status register via MMIO (always safe)
    status = dma.mmio.read(dma_engine.REG_S2MM_DMASR)
    ctrl = dma.mmio.read(dma_engine.REG_S2MM_DMACR)
    
    print("=== DMA Status Register (S2MM_DMASR) ===")
    print(f"  Raw value: 0x{status:08X}")
    print(f"  HALTED:    {bool(status & 0x01)} (bit 0)")
    print(f"  IDLE:      {bool(status & 0x02)} (bit 1)")
    print(f"  SGIncld:   {bool(status & 0x08)} (bit 3)")
    print(f"  DMAIntErr: {bool(status & 0x10)} (bit 4)")
    print(f"  DMASlvErr: {bool(status & 0x20)} (bit 5)")
    print(f"  DMADecErr: {bool(status & 0x40)} (bit 6)")
    print(f"  IOC_Irq:   {bool(status & 0x1000)} (bit 12)")
    print(f"  Dly_Irq:   {bool(status & 0x2000)} (bit 13)")
    print(f"  Err_Irq:   {bool(status & 0x4000)} (bit 14)")
    
    print("\n=== DMA Control Register (S2MM_DMACR) ===")
    print(f"  Raw value: 0x{ctrl:08X}")
    print(f"  RS (Run):  {bool(ctrl & 0x01)} (bit 0)")
    print(f"  Reset:     {bool(ctrl & 0x04)} (bit 2)")
    
    # PYNQ state - only read if safe (DMA not potentially blocked)
    if not safe_mode:
        print("\n=== PYNQ Channel State ===")
        print(f"  recvchannel.running: {dma.recvchannel.running}")
        print(f"  recvchannel.idle:    {dma.recvchannel.idle}")
    else:
        print("\n=== PYNQ Channel State ===")
        print("  [SAFE MODE - skipping PYNQ calls to avoid blocking]")
        # Interpret from MMIO
        hw_running = bool(ctrl & 0x01)  # RS bit
        hw_halted = bool(status & 0x01)
        hw_idle = bool(status & 0x02)
        print(f"  HW Running (RS bit):  {hw_running}")
        print(f"  HW Halted:  {hw_halted}")
        print(f"  HW Idle:    {hw_idle}")
    
    return status, ctrl

# Initial state (safe to use PYNQ here, DMA should be idle)
print(">>> INITIAL DMA STATE (after backend init)")
read_dma_status_mmio(backend.dma_engine, safe_mode=False)

## 4. AXI Switch Routing Verification

The AXI Switch routes data from ADC outputs to the DMA:
- Different ports for raw vs decimated/accumulated modes
- Switch must be re-committed after port changes

In [ ]:
def read_switch_config(dma_engine):
    """Read AXI Switch routing configuration."""
    switch = dma_engine.switch
    if switch is None:
        print("WARNING: No AXI Switch found!")
        return None
    
    ctrl = switch.mmio.read(dma_engine.REG_CTRL)
    mi_mux_0 = switch.mmio.read(dma_engine.REG_MI_MUX_0)
    
    print("=== AXI Switch Configuration ===")
    print(f"  CTRL Register:   0x{ctrl:08X}")
    print(f"  MI_MUX[0]:       0x{mi_mux_0:08X}")
    
    # Decode routing
    # Formula: adc_index * 2 + offset (offset=0 for raw, 1 for decimated/accumulated)
    adc_idx = mi_mux_0 // 2
    mode_offset = mi_mux_0 % 2
    mode_name = "raw" if mode_offset == 0 else "decimated/accumulated"
    
    print(f"\n  Decoded Routing:")
    print(f"    ADC Index: {adc_idx}")
    print(f"    Mode: {mode_name}")
    
    return mi_mux_0

print(">>> INITIAL SWITCH STATE")
read_switch_config(backend.dma_engine)

In [ ]:
# Test switch routing for different modes
print(">>> TESTING SWITCH ROUTING")

test_cases = [
    (0, False, "ADC0 Decimated"),
    (0, True,  "ADC0 Raw"),
    (1, False, "ADC1 Decimated"),
    (1, True,  "ADC1 Raw"),
]

for adc_idx, raw_mode, description in test_cases:
    print(f"\n--- {description} ---")
    backend.dma_engine._route_switch(adc_idx, raw_mode=raw_mode)
    read_switch_config(backend.dma_engine)
    
print("\n[PASS] All routing configurations successful")

## 5. DMA Arm/Abort Cycle Test

Test the DMA arming process and abort WITHOUT triggering acquisition.

**CRITICAL**: When DMA is armed but waiting for data, PYNQ's `stop()` and even reading `.idle` can BLOCK forever. The abort() must use direct MMIO reset.

In [ ]:
print(">>> TEST: DMA Arm/Abort Cycle")
print("    This tests arming DMA without triggering, then aborting.")
print("    If this hangs, abort() is not working correctly.\n")

# Arm for decimated mode
print("1. Arming DMA (decimated, 256 samples, ADC 0)...")
buffer = backend.dma_engine.arm_acquisition(256, 'decimated', adc_index=0)
print(f"   Buffer allocated: shape={buffer.shape}, dtype={buffer.dtype}")

print("\n2. DMA State after arming (MMIO only - safe mode):")
read_dma_status_mmio(backend.dma_engine, safe_mode=True)

print("\n3. Aborting DMA (via MMIO reset, not PYNQ stop)...")
backend.dma_engine.abort()
print("   Abort completed.")

print("\n4. DMA State after abort (MMIO only):")
read_dma_status_mmio(backend.dma_engine, safe_mode=True)

print("\n5. Freeing buffer...")
if hasattr(buffer, 'freebuffer'):
    buffer.freebuffer()
    print("   Buffer freed.")

print("\n[PASS] Arm/Abort cycle completed without blocking")

## 6. Post-Abort Recovery Test

Verify that DMA can be used normally after an abort.

In [ ]:
print(">>> TEST: Post-Abort Recovery")
print("    Running a normal acquisition after abort to verify recovery.\n")

try:
    data = backend.start_experiment(
        duration_cycles=1500,
        readout_cfg={
            'target_label': 'Readout_Line',
            'num_samples': 256,
            'freq': 200.0,
            'mode': 'decimated'
        }
    )
    print(f"[PASS] Acquisition successful: {data.shape[0]} samples")
    print(f"       Mean magnitude: {np.abs(data).mean():.2f}")
except Exception as e:
    print(f"[FAIL] Acquisition failed: {type(e).__name__}: {e}")

## 7. Timeout Handling Test

Test that DMA timeout is properly caught and cleaned up.

In [ ]:
print(">>> TEST: DMA Timeout Handling")
print("   This test arms DMA but does NOT trigger, expecting timeout.")
print("   Expected: DMATimeoutError after ~1 second\n")

# Arm but don't trigger
buffer = backend.dma_engine.arm_acquisition(64, 'decimated', adc_index=0)
print(f"   Buffer armed: {buffer.shape}")

print("   Waiting for timeout (1 second)...")
t_start = time.time()
try:
    # Use very short timeout
    data = backend.dma_engine.retrieve_acquisition(buffer, 'decimated', timeout=1)
    print("   ERROR: Should have timed out!")
except DMATimeoutError as e:
    t_elapsed = time.time() - t_start
    print(f"   [EXPECTED] DMATimeoutError after {t_elapsed:.1f}s")
except Exception as e:
    t_elapsed = time.time() - t_start
    print(f"   [UNEXPECTED] {type(e).__name__} after {t_elapsed:.1f}s: {e}")

print("\n   DMA state after timeout (MMIO):")
read_dma_status_mmio(backend.dma_engine, safe_mode=True)

print("\n[PASS] Timeout properly handled")

## 8. Zombie Killer Test (Transactional Safety)

Test that failed experiments properly clean up DMA state.

In [ ]:
print(">>> TEST: Zombie Killer (Backend Transactional Safety)")

print("\n1. Initial health check:")
health = backend.check_hardware_health()
print(f"   DMA: {health['dma']}")

print("\n2. Attempting experiment with impossibly short duration...")
try:
    data = backend.start_experiment(
        duration_cycles=10,  # Way too short - should fail validation
        readout_cfg={
            'target_label': 'Readout_Line',
            'num_samples': 1000,
            'freq': 200.0,
            'mode': 'decimated'
        },
        skip_validation=False
    )
    print("   ERROR: Should have failed!")
except TimingError as e:
    print(f"   [EXPECTED] TimingError caught")
except Exception as e:
    print(f"   [UNEXPECTED] {type(e).__name__}: {e}")

print("\n3. Verify DMA can still be used:")
try:
    data = backend.start_experiment(
        duration_cycles=2000,
        readout_cfg={
            'target_label': 'Readout_Line',
            'num_samples': 100,
            'freq': 200.0,
            'mode': 'decimated'
        }
    )
    print(f"   [PASS] Got {data.shape[0]} samples - DMA recovered properly")
except Exception as e:
    print(f"   [FAIL] {type(e).__name__}: {e}")

## 9. Consecutive Acquisitions Test

In [ ]:
print(">>> TEST: Consecutive Acquisitions")

N_ITERATIONS = 5
results = []

for i in range(N_ITERATIONS):
    try:
        data = backend.start_experiment(
            duration_cycles=1500,
            readout_cfg={
                'target_label': 'Readout_Line',
                'num_samples': 256,
                'freq': 200.0,
                'mode': 'decimated'
            }
        )
        results.append('OK')
        print(f"   [{i+1}/{N_ITERATIONS}] OK - {data.shape[0]} samples")
    except Exception as e:
        results.append('FAIL')
        print(f"   [{i+1}/{N_ITERATIONS}] FAIL - {e}")

successes = results.count('OK')
print(f"\nPassed: {successes}/{N_ITERATIONS}")
if successes == N_ITERATIONS:
    print("[PASS] All consecutive acquisitions successful")
else:
    print("[FAIL] Some acquisitions failed")

## 10. Mode Switching Test

In [ ]:
print(">>> TEST: Mode Switching")

test_modes = [
    ('raw', 512),
    ('raw', 512),
    ('accumulated', 128),
    ('decimated', 256),
    ('decimated', 256),
    ('accumulated', 128),
]

for mode, n_samples in test_modes:
    try:
        data = backend.start_experiment(
            duration_cycles=2000,
            readout_cfg={
                'target_label': 'Readout_Line',
                'num_samples': n_samples,
                'freq': 200.0,
                'mode': mode
            }
        )
        print(f"   {mode:12} -> {data.shape[0]} samples [OK]")
    except Exception as e:
        print(f"   {mode:12} -> FAILED: {e}")

print("\n[PASS] Mode switching test complete")

## 11. Final Health Check

In [ ]:
print(">>> FINAL HEALTH CHECK")

health = backend.check_hardware_health()

print(f"\nDMA Status: {health['dma']}")
print(f"Clocks: {health['clocks']}")
print(f"Trigger: {health['trigger']}")

print("\nGenerators:")
for g in health['generators']:
    print(f"  [{g['index']}] {g['status']}")

print("\nAcquisitions:")
for a in health['acquisitions']:
    print(f"  [{a['index']}] {a['status']}")

print("\n" + "="*50)
print("DMA DEBUG NOTEBOOK COMPLETE")
print("="*50)